# Imports

In [ ]:
#standard imports
from dotenv import load_dotenv
import zipfile
from collections import defaultdict
from pathlib import Path

#API imports
from kaggle.api.kaggle_api_extended import KaggleApi

#database and efficient data handling imports
import duckdb
import pyarrow as pa
import pyarrow.compute as pc

#general data manipulation and visualization imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#machine learning and model optimisation
import lightgbm as lgb
import optuna
from optuna.integration import LightGBMPruningCallback

#model evaluation and explainability imports
from sklearn.metrics import precision_recall_curve, roc_auc_score
import shap

# Load Data

In [ ]:
load_dotenv() #loads environment variables from .env file

In [ ]:
data_folder = Path(data) #where the data will be stored
sql_folder = Path(sql) #where the SQL code is stored

zip_path = data_folder / 'home-credit-default-risk.zip' #where the zip file will be downloaded

In [ ]:
if not any(data_folder.glob('*.csv')): #if .csv files do not exist then download the data from Kaggle
    print(f"Fetching data from Kaggle into {data_folder}...")

    api = KaggleApi() #initialises Kaggle API client object
    api.authenticate() #loads credentials from .env file to the Kaggle API client object
    api.competition_download_files('home-credit-default-risk', path=data_folder) #local Kaggle API client object pings the Kaggle API server to download the data from the competition into the data folder

    print("Extracting files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref: #reads .zip
        zip_ref.extractall(data_folder) #unzips the files into the data folder
        
    zip_path.unlink(missing_ok=True) #deletes the .zip file after extraction
else:
    print(f"Data already exists locally in {data_folder}. Skipping download.") #if data already exists locally, skip the download step


print("✅ Files downloaded")
for file in data_folder.iterdir(): #iterates through the files in the data folder
    print(file.name) #prints file names in the data folder

# Create and Populate DuckDB Instance

In [ ]:
db = duckdb.connect() #connects to an in-memory DuckDB database instance - this acts like a "remote control" to the DuckDB database engine

In [ ]:
data_ignore = ("description", "sample") #ignore tables for submission template/column descriptions as not needed for analysis

for csv_file in data_folder.glob("*.csv"): #across all downloaded files

  if any(keyword in csv_file.name.lower() for keyword in data_ignore): #skip files that are not needed
        continue

  table_name = csv_file.stem.lower() #define table name from file name

#create a DuckDB table from the CSV file using read_csv_auto

  query = f"""
  CREATE TABLE IF NOT EXISTS {table_name} AS
  SELECT *
  FROM read_csv_auto('{csv_file}');
  """
  print(f"Creating table: {table_name}")
  db.execute(query) #create the table

# Initial Data Exploration

In [ ]:
tables = db.sql("SHOW TABLES").fetchall()
print(tables) #outputs loaded tables in the DuckDB database

for table in tables:
    
    table_name = table[0] #get string name of the table from the tuple returned by SHOW TABLES

    num_rows = db.sql(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0] #get number of rows in the table using COUNT(*)
    num_cols = len(db.sql(f"DESCRIBE {table_name}").fetchall()) #get number of columns in the table using DESCRIBE and counting the number of rows returned

    print(f"\n{'='*100}")
    print(f"📅 Table: {table_name} | Shape: ({num_rows} rows, {num_cols} features)") #print header for the table with its shape
    print(f"{'='*100}")

    print("Top 10 Rows:")
    display(db.sql(f"SELECT * FROM {table_name} LIMIT 10").df()) #display the top 10 rows of the table using LIMIT 10
    print()


# combine DESCRIBE and SUMMARIZE results for each table to get a comprehensive view of the table metadata
    combined_query = f"""
    SELECT
        d.column_name,
        d.column_type,
        s.null_percentage,
        s.approx_unique,
        s.min,
        s.max
    FROM (DESCRIBE {table_name}) AS d
    JOIN (SUMMARIZE {table_name}) AS s
      ON d.column_name = s.column_name
    ORDER BY s.null_percentage DESC;
    """

    print(f"Combined Metadata:")
    display(db.sql(combined_query).df()) #display the combined metadata for the table
    print() #print blank line for better readability

# Create Views

In [ ]:
sql_files = sorted(sql_folder.glob("*.sql")) #list of all .sql files in the sql folder, sorted alphabetically (to order 01, 02, 03, ..., 09)

for sql_code_file in sql_files[:-1]: #all but last, 09, since this needs python string input and can not be created by itself
    print(f"Building view from {sql_code_file.name}...")
    query = sql_code_file.read_text(encoding='utf-8')
    db.execute(query) #build view

# Build Master Dataset

In [ ]:
def build_master_dataset(db, source_table, output_table): #source_table is application_test or application_train, output_table is the name to give the table that will be made in DuckDB e.g. master_train_all or master_test_all
    """
    Reads the parameterized SQL template defined in 09_master_dataset.sql, injects the table names defined in this function's arguments, executes the join efficiently in DuckDB, and returns an Apache Arrow table
    """
    #open and read the SQL template
    with open('sql/09_master_dataset.sql', 'r') as file:
        sql_template = file.read()

    #inject the source and output table names into the SQL template
    query = sql_template.format(output_table=output_table, source_table=source_table)

    print(f"Aggregatting and joining tables up the chain for {source_table}...")
    db.execute(query)

    print(f"✅ Successfully created DuckDB table '{output_table}'. Extracting to Apache Arrow...")

    #return the newly created table directly into an Apache Arrow table
    return db.sql(f"SELECT * FROM {output_table}").arrow()

In [ ]:
#build master datasets for training and testing using the function defined above
train_data = build_master_dataset(db=db, source_table="application_train", output_table="master_train_all")
test_data = build_master_dataset(db=db, source_table="application_test", output_table="master_test_all")

In [ ]:
print(f"{train_data.num_rows} rows and {train_data.num_columns} columns in train data.")
print(f"{test_data.num_rows} rows and {test_data.num_columns} columns in test data.")

Note there is one fewer fields in test data (`TARGET`)

In [ ]:
top_5_table = train_data.slice(0, 5) #get top 5 rows of apache table
top_5_table = top_5_table.to_pandas() #convert to pandas
pd.set_option('display.max_columns', None) #allow for large display
display(top_5_table) #display table

# Down/Re-Cast Field Datatypes

In [ ]:
metadata_rows = []

for field in train_data.schema:
  col_name = field.name
  col_type = str(field.type)
  null_count = train_data[col_name].null_count
  metadata_rows.append({"Column Name": col_name, "Data Type": col_type, "Null Count": null_count})

df_meta = pd.DataFrame(metadata_rows)
display(df_meta)

Above visualises all features in the master dataset outputted from the SQL aggregation pipeline. Fields expectedly vary in datatype, however, not all are compatible with LightGBM.

The datatypes seen in the training data are:
- `bool`
- `int64`
- `int32`
- `string`
- `double`
- `decimal128(38,0)`

LightGBM works optimally with 32 bit data due to its underlying architecture. Since this is a large dataset and there are no fields which require significant precision, field datatypes should be downcast to 32-bit versions where appropriate.

However, blindly recasting field datatypes is not solely appropriate since other data transformations are needed, such as for categorical variable encoding.

Inspection of the fields show that boolean features exist across `int64`, `int32`, and `string` fields. `string` fields need special handling since their True/False values are not simply an integer version of 1/0.

`double` and `decimal128(38,0)` fields can be mass downcast to float32.

The following defines various helper functions to deterministically complete this processing, regardless of the dataset (train/test) being passed, aggregated by a final `process_data()` function.

In [ ]:
def get_table_metadata(table):
    columns_by_type = defaultdict(list)

    for field in table.schema:
        col_name = field.name
        col_type = str(field.type)
        columns_by_type[col_type].append(col_name)

    return dict(columns_by_type)

In [ ]:
def recast_arrow_table(table, to_cat_cols, to_int32_cols, to_float32_cols, to_bool_cols):

    # Step 1: Dictionary encode categoricals (requires operating on data arrays)
    for col in to_cat_cols:
        idx = table.schema.get_field_index(col)
        col_data = table.column(col)
        if not pa.types.is_dictionary(col_data.type):
            table = table.set_column(idx, col, col_data.dictionary_encode())

    # Step 2: Build a new schema for the numeric and boolean columns
    new_fields = []
    for field in table.schema:
        if field.name in to_int32_cols and not pa.types.is_int32(field.type):
            new_fields.append(pa.field(field.name, pa.int32()))
        elif field.name in to_float32_cols and not pa.types.is_float32(field.type):
            new_fields.append(pa.field(field.name, pa.float32()))
        elif field.name in to_bool_cols and not pa.types.is_boolean(field.type):
            new_fields.append(pa.field(field.name, pa.bool_())) # Note: pa.bool_() is the PyArrow boolean type
        else:
            new_fields.append(field)  # Keep original type

    new_schema = pa.schema(new_fields)

    # Step 3: Cast the entire table to the new schema in one operation
    return table.cast(new_schema)

In [ ]:
def cast_string_to_bool(tbl, col_name, true_string):
    # 1. Compute the boolean array
    bool_array = pc.equal(tbl[col_name], true_string)

    # 2. Find the column's current index
    col_idx = tbl.schema.get_field_index(col_name)

    # 3. Replace the column in the table
    return tbl.set_column(col_idx, col_name, bool_array)

In [ ]:
def process_data(table):
  datatype_dict = get_table_metadata(table)


  #lists of strings of the fields of each datatype
  int64_cols = datatype_dict["int64"]
  int32_cols = datatype_dict["int32"]

  double_cols = datatype_dict["double"]
  decimal_cols = datatype_dict["decimal128(38, 0)"]

  string_cols = datatype_dict["string"]

  bool_cols = datatype_dict["bool"]

  #original string columns
  string_to_bool = (
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY")

  string_true_value = ["M", "Y", "Y"]

  to_cat_cols = [item for item in string_cols if item not in string_to_bool] #category columns are all those in string except those three in the list, which can instead be cast to bool

  for field, value in zip(string_to_bool, string_true_value):
    table = cast_string_to_bool(table, field, value)

  #original int64 columns
  int64_to_bool = (
    "TARGET",
    "FLAG_MOBIL",
    "FLAG_EMP_PHONE",
    "FLAG_WORK_PHONE",
    "FLAG_CONT_MOBILE",
    "FLAG_PHONE",
    "FLAG_EMAIL",
    "REG_REGION_NOT_LIVE_REGION",
    "REG_REGION_NOT_WORK_REGION",
    "LIVE_REGION_NOT_WORK_REGION",
    "REG_CITY_NOT_LIVE_CITY",
    "REG_CITY_NOT_WORK_CITY",
    "LIVE_CITY_NOT_WORK_CITY",
    "FLAG_DOCUMENT_2",
    "FLAG_DOCUMENT_3",
    "FLAG_DOCUMENT_4",
    "FLAG_DOCUMENT_5",
    "FLAG_DOCUMENT_6",
    "FLAG_DOCUMENT_7",
    "FLAG_DOCUMENT_8",
    "FLAG_DOCUMENT_9",
    "FLAG_DOCUMENT_10",
    "FLAG_DOCUMENT_11",
    "FLAG_DOCUMENT_12",
    "FLAG_DOCUMENT_13",
    "FLAG_DOCUMENT_14",
    "FLAG_DOCUMENT_15",
    "FLAG_DOCUMENT_16",
    "FLAG_DOCUMENT_17",
    "FLAG_DOCUMENT_18",
    "FLAG_DOCUMENT_19",
    "FLAG_DOCUMENT_20",
    "FLAG_DOCUMENT_21")

  int64_to_int32 = [item for item in int64_cols if item not in int64_to_bool]


  #original int32 columns
  int32_to_bool = (
    "flag_employed_anomaly",
    "b_any_current_debt",
    "b_any_currently_overdue",
    "bb_any_historical_dpd",
    "bb_any_closed_history",
    "bb_any_unknown_history",
    "bb_any_severe_default",
    "bb_any_balance_history_available",
    "pa_most_recent_app_was_rejected",
    "pos_any_past_due_last_12m",
    "pos_any_term_change_ever",
    "cc_any_atm_draw_ever",
    "cc_any_late_fees_ever")


  to_float_32 = double_cols + decimal_cols
  to_bool_cols = int64_to_bool + int32_to_bool

  table = recast_arrow_table(table, to_cat_cols, int64_to_int32, to_float_32, to_bool_cols)

  return table, to_cat_cols

In [ ]:
data_train_clean, cat_cols = process_data(train_data) #apache arrow table
data_test_clean, _ = process_data(test_data) #apache arrow table

In [ ]:
clean_train_data_dict = get_table_metadata(data_train_clean) #field datatype makeup dict
clean_test_data_dict = get_table_metadata(data_test_clean) #field datatype makeup dict

Above shows a visualisation of the dictionary structure used before, with keys being string representations of the datatypes existing within the Apache Arrow table, and values being a list of strings of the respective field names.

In [ ]:
for key, values in clean_train_data_dict.items():
    print(f"{key} // {len(values)}")
    print("-" * 20)
    for value in values:
        print(value)
    print("-" * 20)

In [ ]:
for key, values in clean_test_data_dict.items():
    print(f"{key} // {len(values)}")
    print("-" * 20)
    for value in values:
        print(value)
    print("-" * 20)

The number of fields per datatype match the expected counts, confirming the transformation has been successful. Additional inspection shows that boolean fields have been cast to `bool`, dictionary encoding exists for the categoric variables, and `double` and `decimal128(38,0)` datatypes are no longer present.

# int32 Mappings for GPU

The current datasets use dictionary encoding for categorical variables, which is supported by CPU-based LightGBM but incompatible with GPU execution which can only process `int`, `bool`, and `float` columns.

To resolve this, the following transformations extract the integer indicies and mappings from the dictionary datatype of the training data, reformmating the column to only contain integer representations of the categories.

Extracting the training mappings is done to apple the same mapping to the test data, preventing data leakage and ensuring reproducability with equivalent indicies. If a category unseen in the training data exists in the test data, it will be replaced with NaN.

In [ ]:
def extract_and_convert_train(train_table):
    """
    Identifies dictionary columns, extracts their integer indices,
    and saves the categorical mappings for test data.
    """
    mappings = {}
    new_arrays = [] #actual data
    new_fields = [] #field name and datatype

    for field in train_table.schema:
        col_name = field.name
        col_data = train_table[col_name].combine_chunks()

        if pa.types.is_dictionary(col_data.type):
            # Save the unique categorical values (the mapping)
            mappings[col_name] = col_data.dictionary #e.g. adding unique categories of a column name (key) to mappings with value: ["cat0", "cat1", ...]
            int_array = col_data.indices #e.g. [0, 1, 1, 0, 2]

            # Keep only the int32 indices for LightGBM
            new_fields.append(pa.field(col_name, int_array.type)) #define field name and datatype
            new_arrays.append(int_array) #define the data of the field
        else:
            # Pass non-dictionary columns through unchanged
            new_fields.append(field)
            new_arrays.append(col_data)

    processed_table = pa.Table.from_arrays(new_arrays, schema=pa.schema(new_fields)) #recreate arrow table, schema defined by the new_fields list (contains field name and datatype), and the respective data from new_arrays
    return processed_table, mappings #returns both table and the mappings used on the data

In [ ]:
master_train_all, train_mappings = extract_and_convert_train(data_train_clean)

In [ ]:
def apply_train_mapping_to_test(test_table, mappings):
    """
    Applies the training mappings to test data.
    Unseen categories automatically become `null`.
    """
    new_arrays = [] #actual data
    new_fields = [] #field name and datatype

    for field in test_table.schema:
        col_name = field.name
        col_data = test_table[col_name].combine_chunks()

        if col_name in mappings:
            # If the test column is also a PyArrow dictionary, flatten it to strings first
            if pa.types.is_dictionary(col_data.type):
                col_data = pc.dictionary_decode(col_data) #flattends back to string categories based on encoded dictionary

            # Map the test strings to the exact integer indices from the train set
            # Categories not found in the value_set automatically return null
            train_categories = mappings[col_name] #list of categories that model saw in training
            mapped_indices = pc.index_in(col_data, value_set=train_categories) #maps string categories to the appropriate index seen in training, if not seen then null

            new_fields.append(pa.field(col_name, mapped_indices.type))
            new_arrays.append(mapped_indices)
        else:
            new_fields.append(field)
            new_arrays.append(col_data)

    processed_table = pa.Table.from_arrays(new_arrays, schema=pa.schema(new_fields)) #recreate arrow table, schema defined by the new_fields list (contains field name and datatype), and the respective data from new_arrays

    return processed_table

In [ ]:
master_test_all = apply_train_mapping_to_test(data_test_clean, train_mappings)

These tables have the suffix `_all` since they contain all features. Subsequent analyses will drop features based on noise probe thresholds and potentially Spearman correlation.

# Probe Feature/Random Bar Injection

At this point, train and test datasets are comprised of many features. Some of these will contain strong predictive signals, whereas others will be indistinguishable from noise.

To form the most simple and explainable LightGBM model whilst retaining predictive power, some features should be pruned from the model.

Instead of setting an arbitrary threshold, such as "include top 50 features", a quantitative threshold can be derived by injecting features with random synthetic noise. Their SHAP values, which indicate feature importance to a model's prediction, can be inspected to derive a cut off where a feature provides meaningful signal compared to noise.

Each feature shall be probed by different noise distributions based on their datatype. This allows appropriate thresholds to be set for each since different datatypes may need different cut offs.

Continuous data is probed via gaussian and uniform distributions, categoric data via random integers, booleans via bernoulli, and count via poisson.

Lists of column names across each datatype can be derived as follows.

In [ ]:
bool_cols  = [field.name for field in master_train_all.schema if field.type == pa.bool_()] #all fields with boolean datatype are boolean

contin_cols = [field.name for field in master_train_all.schema if field.type == pa.float32()] #all fields with float32 datatype are continuous

#cat_cols returned before by process_data()

count_cols = [field.name for field in master_train_all.schema if field.type == pa.int32() and field.name not in cat_cols] #count columns are those with int32 datatype, excluding the categorical columns

print(f"Boolean Columns: {bool_cols}")
print(f"Continuous Columns: {contin_cols}")
print(f"Categoric Columns: {cat_cols}")
print(f"Count Columns: {count_cols}")
print()
print(f"Total fields included in train are {len(bool_cols + contin_cols + cat_cols + count_cols)}")

This confirms that all features, including `TARGET`, are accounted for.

In [ ]:
def inject_noise_probes(table: pa.Table, n_continuous: int, n_categorical: int, n_boolean: int, n_count: int, random_state: int = 42,) -> pa.Table:
    """
    Inject random noise probe features into an Apache Arrow table.
    """

    rng = np.random.default_rng(random_state)
    n_rows = table.num_rows

    # Continuous probes: alternate between Gaussian and Uniform
    for i in range(n_continuous):
        if i % 2 == 0:
            noise = rng.normal(0, 1, n_rows)
            distribution = "gaussian"
        else:
            noise = rng.uniform(0, 1, n_rows)
            distribution = "uniform"

        table = table.append_column(f"probe_continuous_{distribution}_{i}", pa.array(noise, type=pa.float32()))

    # Categorical probes
    for i in range(n_categorical):
        noise = rng.integers(0, 5, n_rows) #setting to 5 since roughly how many features exist per categoric variable, could extend to make distinct noise probes for each number of categories
        table = table.append_column(f"probe_categorical_{i}", pa.array(noise, type=pa.int32()))

    # Boolean probes: Bernoulli = Binomial(n=1)
    for i in range(n_boolean):
        noise = rng.binomial(n=1, p=0.5, size=n_rows,).astype(bool) #note that only considering p=0.5 i.e. on average 50% True 50% false, could extend this by having varied p values and mapping to different features based on T/F occurence rate (e.g. use a noise probe w/ p=0.1 for a feature which shows true ~10% of the time)
        table = table.append_column(f"probe_boolean_{i}", pa.array(noise, type=pa.bool_()))

    # Count probes: Poisson
    for i in range(n_count):
        noise = rng.poisson(lam=5, size=n_rows) #note can vary expected value, lam, to create more specific probes for certain features with different expected values
        table = table.append_column(f"probe_count_poisson_{i}", pa.array(noise, type=pa.int32()))

    return table

In [ ]:
probed_baseline_train = inject_noise_probes(master_train_all, n_continuous=5, n_categorical=5, n_boolean=5, n_count=5, random_state=42)
probed_baseline_test = inject_noise_probes(master_test_all, n_continuous=5, n_categorical=5, n_boolean=5, n_count=5, random_state=42)

# Baseline LightGBM

In [ ]:
def arrow_train_test_split(table, target_col, test_frac, random_state):

    n_rows = table.num_rows
    rng = np.random.default_rng(random_state)

    is_train = rng.uniform(0.0, 1.0, size=n_rows) >= test_frac
    train_mask = pa.array(is_train)
    test_mask = pc.invert(train_mask)


    X_table = table.drop([target_col])
    y_table = table.select([target_col])

    X_train = pc.filter(X_table, train_mask)
    X_test = pc.filter(X_table, test_mask)
    y_train = pc.filter(y_table, train_mask)["TARGET"].to_numpy()
    y_test = pc.filter(y_table, test_mask)["TARGET"].to_numpy()

    return X_train, X_test, y_train, y_test

In [ ]:
def run_lightgbm(train_data, cat_features, bparams=None):

  X_train, X_val, y_train, y_val = arrow_train_test_split(train_data, "TARGET", 0.2, 42)

  dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
  dval = lgb.Dataset(X_val, label=y_val, reference=dtrain, categorical_feature=cat_features)

  total_negative = np.sum(y_train == 0)
  total_positive = np.sum(y_train == 1)
  pos_weight = total_negative / total_positive

  print(f"TARGET Imbalance: {pos_weight:.2f}:1")
  if bparams is None:
    bparams = {
      'objective': 'binary',
      'metric': 'average_precision',
      'scale_pos_weight': pos_weight,
      'learning_rate': 0.01,
      'verbose': -1,
      'device':'cuda',
      'max_bin': 63,
      'min_data_in_leaf': 250,
      'feature_fraction': 0.5,
      'bagging_fraction': 0.5,
      'bagging_freq': 5,
      'lambda_l1': 0.1,
      'lambda_l2': 1.0}

  best_val_ap = 0
  best_model = None
  best_leaves = None
  best_evals_result = None

  for leaves in [31, 63, 127]:
    params = bparams.copy()
    params['num_leaves'] = leaves

    evals_result = {}

    model = lgb.train(
          params,
          dtrain,
          valid_sets=[dtrain, dval], #track both train and validation set metrics
          valid_names=['train', 'valid'], #identify dataset names for above
          num_boost_round=500,
          callbacks=[lgb.early_stopping(stopping_rounds=15), lgb.record_evaluation(evals_result)])

    current_val_ap = model.best_score['valid']['average_precision']

    if current_val_ap > best_val_ap:
      best_val_ap = current_val_ap
      best_model = model
      best_leaves = leaves
      best_evals_result = evals_result

  print(f"Best Validation Average Precision: {best_val_ap} (with {best_leaves} leaves)")

  #plot the average_precision metric history for the best model
  lgb.plot_metric(best_evals_result, metric='average_precision', title='Model Performance over Boosting Rounds')
  plt.show()


  print("Using validation set to get metrics...") #since test set doesn't have published target

  X_val_df = X_val.to_pandas()
  object_cols = X_val_df.select_dtypes(include=['object']).columns
  X_val_df[object_cols] = X_val_df[object_cols].astype('float32')
  val_predictions = best_model.predict(X_val_df)

  precision, recall, thresholds = precision_recall_curve(y_val, val_predictions)

  f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)

  best_index = np.argmax(f1_scores)

  best_f1 = f1_scores[best_index]
  best_precision = precision[best_index]
  best_recall = recall[best_index]
  best_threshold = thresholds[best_index]

  #calculate model assessment metrics
  roc_auc = roc_auc_score(y_val, val_predictions)

  print(f"Optimal Threshold: {best_threshold:.4f}")
  print(f"Max F1 Score:      {best_f1:.4f}")
  print(f"Precision at Max:  {best_precision:.4f}")
  print(f"Recall at Max:     {best_recall:.4f}")
  print(f"ROC AUC Score:     {roc_auc:.4f}")



  X_val_sample = X_val_df.sample(n=10000, random_state=42)
  explainer = shap.TreeExplainer(best_model)
  shap_values = explainer(X_val_sample)

  shap.summary_plot(shap_values, X_val_sample, show=False)

  title_text = f"SHAP Summary Plot\nBest F1: {best_f1:.2f} | ROC AUC: {roc_auc:.2f} | AP: {best_val_ap:.2f}"
  plt.title(title_text, fontsize=14, pad=15)

  plt.show()

  mean_abs_shap = np.abs(shap_values.values).mean(axis=0) #shap_values is a matrix of same dimensions as the data. one field per feature, one record per observation. outputs an array of absolute mean shap value for a given field

  shap_importance = pd.Series(mean_abs_shap, index=shap_values.feature_names, name="mean_abs_shap",).sort_values(ascending=False) #convert to series object with index being the string representation of feature names

  display(shap_importance.head(19))

  return best_model, shap_importance, pos_weight

In [ ]:
probed_baseline_model, probed_baseline_shap_importance, pos_weight_ = run_lightgbm(probed_baseline_train, cat_cols)

# Probe Thresholds (SHAP)

The SHAP values have been generated for the baseline model which includes the noise probes. Therefore, these can be interrogated to get the SHAP value thresholds for each type of feature in the master datasets.

Any features falling below the threshold are dropped.

In [ ]:
#list of strings of probe feature names

continuous_probes = [name for name in probed_baseline_shap_importance.index if name.startswith("probe_continuous_")]

categorical_probes = [name for name in probed_baseline_shap_importance.index if name.startswith("probe_categorical_")]

boolean_probes = [name for name in probed_baseline_shap_importance.index if name.startswith("probe_boolean_")]

count_probes = [name for name in probed_baseline_shap_importance.index if name.startswith("probe_count_")]

print(f"Continuous Probes: {continuous_probes}")
print(f"Categorical Probes: {categorical_probes}")
print(f"Boolean Probes: {boolean_probes}")
print(f"Count Probes: {count_probes}")
print()
print(f"Total fields included are {len(continuous_probes + categorical_probes + boolean_probes + count_probes)}") #for validation

In [ ]:
#define thresholds for each probe type
cutoff_percentile = 0.95

thresholds = {
    "continuous": probed_baseline_shap_importance.loc[continuous_probes].quantile(cutoff_percentile), #takes percentile shap value from the continuous subset of shap_importance
    "categorical": probed_baseline_shap_importance.loc[categorical_probes].quantile(cutoff_percentile),
    "boolean": probed_baseline_shap_importance.loc[boolean_probes].quantile(cutoff_percentile),
    "count": probed_baseline_shap_importance.loc[count_probes].quantile(cutoff_percentile)}

In [ ]:
bool_cols_no_target = [feature for feature in bool_cols if feature != "TARGET"]

print(f"Boolean Columns: {bool_cols_no_target}")
print(f"Continuous Columns: {contin_cols}")
print(f"Categoric Columns: {cat_cols}")
print(f"Count Columns: {count_cols}")
print()

feature_groups = {
    "continuous": contin_cols,
    "categorical": cat_cols,
    "boolean": bool_cols_no_target,
    "count": count_cols}

features_to_drop = {}

for feature_type, feature_names in feature_groups.items():

    cutoff = thresholds[feature_type]
    dropped_features = []

    for feature_name in feature_names:
      importance = probed_baseline_shap_importance[feature_name] #access via string index defined before

      if importance <= cutoff:
        dropped_features.append(feature_name) #if shap below threshold then add to drop list

    features_to_drop[feature_type] = dropped_features #new key for the category with values being list of features to drop


print("Features to drop (dict):")
print(features_to_drop)

probe_col_to_drop = []

for feature_type, features in features_to_drop.items():
    probe_col_to_drop.extend(features)

print()
print(f"{len(probe_col_to_drop)} features to drop:")
print(*sorted(probe_col_to_drop), sep='\n')

In [ ]:
master_train_denoise = master_train_all.drop_columns(probe_col_to_drop)
master_test_denoise = master_test_all.drop_columns(probe_col_to_drop)

In [ ]:
denoise_cat_cols = [col for col in cat_cols if col not in probe_col_to_drop]

In [ ]:
denoise_model, denoise_shap_importance, _ = run_lightgbm(master_train_denoise, denoise_cat_cols)

# Spearman Correlation

Whilst not always neccesary here since multicollinearity does not invalidate LightGBM models, pruning highly correlated features can make models simpler, more explainable, with greater SHAP value stability.

The trade-off is potential performance degredation. As such, highly correlated features will be identified and dropped with model performance compared. If only a minor degredation in performance, features can be dropped. Otherwise, the correlated features can persist to enable optimal performance.

In [ ]:
def find_correlated_features_to_drop(table: pa.Table, shap_importance, threshold = 0.90, sample_size = 100000, random_state = 42):

  table = table.drop_columns(["TARGET"])

  if table.num_rows > sample_size:
    rng = np.random.default_rng(random_state)
    row_indices = rng.choice(table.num_rows, size=sample_size,  replace=False)
    sampled_table = table.take(row_indices)

  else:
    sampled_table = table

  df = sampled_table.to_pandas()

  correlation_matrix = df.corr(method="spearman").abs()

  ordered_features = sorted(table.column_names, key=lambda feature: shap_importance[feature], reverse=True)

  kept_features = []
  features_to_drop = []

  for feature in ordered_features:
    highly_correlated_with_kept_feature = False

    for kept_feature in kept_features:
      correlation = correlation_matrix.loc[feature, kept_feature]

      if correlation >= threshold:
        highly_correlated_with_kept_feature = True
        break

    if highly_correlated_with_kept_feature:
      features_to_drop.append(feature)

    else:
      kept_features.append(feature)

  return features_to_drop

In [ ]:
highly_correlated_features_to_drop = find_correlated_features_to_drop(master_train_denoise, denoise_shap_importance, threshold=0.90, sample_size=100000, random_state=42)

In [ ]:
print(highly_correlated_features_to_drop)
print(len(highly_correlated_features_to_drop))

In [ ]:
master_train_denoise_uncorr = master_train_denoise.drop_columns(highly_correlated_features_to_drop)
master_test_denoise_uncorr = master_test_denoise.drop_columns(highly_correlated_features_to_drop)

print(f"Before: {master_train_denoise.shape}")
print(f"After: {master_train_denoise_uncorr.shape}")

In [ ]:
denoise_uncorr_cat_cols = [col for col in denoise_cat_cols if col not in highly_correlated_features_to_drop]

In [ ]:
denoise_uncorr_model, denoise_uncorr_shap_importance, _ = run_lightgbm(master_train_denoise_uncorr, denoise_uncorr_cat_cols)

# Protected Attributes

The original training dataset had ~280 features. Around 2/3 of these were pruned by comparing SHAP values with noise probes, and a further few removed as they were highly correlated with a remaining feature with higher SHAP value, leaving around 80 features. Removing these ~200 features did not significantly degrade performance, leaving a simpler and more explainable prediction model.

However, UK regulations, such as the 2010 Equality Act, FCA's Consumer Duty principle, and GDPR, legally forbid the use of protected attributes to promote fairness and combat discriminatory decision making processes.

Prottected attributes are those which contain personal and/or demographic information, whether directly or indirectly as a proxy variable. Models can learn to use these features to derive a predictive signal so should be dropped before models are trained.

Whilst this will degrade model performance, there is no point in having a better model that can' be deployed since it is discriminatory and an unexplainable "black-box".

The majority of the original ~280 features are allowed since they provide unbiased behavioural signals, but protected variables must be identified and dropped.

**Protected Attributes to Remove**


- **Socio-economic proxies:**
  - All housing quality metrics e.g. those ending with `_AVG`, `_MODE`, or `_MEDI`
  - Geographic location metrics e.g. `REGION_RATING_CLIENT` and `REGION_RATING_CLIENT_WITH_CITY`
  - Population density metrics e.g. `REGION_POPULATION_RELATIVE`
  - Acquired Education e.g. `NAME_EDUCATION_TYPE` and by association `index_wealth_stability`, which is derived from the prior


- **Personal Features:**
  - Sex e.g. `CODE_GENDER`
  - Age e.g. `DAYS_BIRTH` and engineered features that leak age (`age_in_years` and `ratio_life_employed`)
  - Marital Status e.g. `NAME_FAMILY_STATUS`
  - Maternity/Paternity e.g. `CNT_CHILDREN` and `CNT_FAM_MEMBERS`

- **Guilt by Association:**
  - Social circle metrics e.g. `OBS_30_CNT_SOCIAL_CIRCLE`, `OBS_60_CNT_SOCIAL_CIRCLE`, `DEF_30_CNT_SOCIAL_CIRCLE`, and `DEF_60_CNT_SOCIAL_CIRCLE` where decisions are not based solely on the individual

**A Note on Retained Features:**
* **Geographic Mismatches:** Flags mapping discrepancies between live/work locations (e.g., `REG_REGION_NOT_LIVE_REGION`) are kept as behavioral/transiency indicators rather than demographic proxies.
* **External Scores:** Aggregated bureau scores (`EXT_SOURCE_1`, `EXT_SOURCE_2`, and `EXT_SOURCE_3`) are highly predictive and chosen to be kept. Whilst these "black boxes" may contain systemic biases, they are industry-standard risk metrics which contain significant predictive signal that cannot be decomposed here. These are analogous to credit report derives scores provided by CRAs such as Experian, Equifax, and TransUnion.

In [ ]:
all_cols_no_target = [feature for feature in master_train_all.column_names if feature != "TARGET"]

denoise_uncorr_cols = master_train_denoise_uncorr.column_names

print(denoise_uncorr_cols)
print(len(denoise_uncorr_cols))

In [ ]:
housing_cols = [feature for feature in all_cols_no_target if feature.endswith(('_AVG', '_MODE', '_MEDI'))] #those that still persist
association_cols = [feature for feature in all_cols_no_target if feature.endswith('_CNT_SOCIAL_CIRCLE')] #those that still persist
print(f"Housing:\n{housing_cols}")
print(f"Association:\n{association_cols}")

In [ ]:
other_protected_attributes = ["CODE_GENDER", "DAYS_BIRTH", "age_in_years", "ratio_life_employed",
                              "NAME_EDUCATION_TYPE", "index_wealth_stability","REGION_POPULATION_RELATIVE", "REGION_RATING_CLIENT",
                              "REGION_RATING_CLIENT_WITH_CITY", "NAME_FAMILY_STATUS", "CNT_CHILDREN", "CNT_FAM_MEMBERS"]

In [ ]:
all_potential_protected_attributes = housing_cols + association_cols + other_protected_attributes
print(len(all_potential_protected_attributes))

In [ ]:
protected_attributes_to_drop = [feature for feature in denoise_uncorr_cols if feature in all_potential_protected_attributes] #protected attributes that persist through denoise and uncorrelation steps
print(f"There are {len(protected_attributes_to_drop)} predictive features to drop for fairness which haven't been dropped based on SHAP noise probes/correlation:\n{protected_attributes_to_drop}")

protected_attr_already_dropped = [col for col in denoise_uncorr_cols if col not in all_potential_protected_attributes] #protected attributes that were removed through denoise and uncorrelation steps

In [ ]:
final_train = master_train_denoise_uncorr.drop_columns(protected_attributes_to_drop)
final_test = master_test_denoise_uncorr.drop_columns(protected_attributes_to_drop)

print(f"Final train shape: {final_train.shape}")
print(f"Final test shape: {final_test.shape}")

In [ ]:
final_cat_cols = [col for col in denoise_uncorr_cat_cols if col not in protected_attributes_to_drop]

In [ ]:
final_model, final_shap_importance, _ = run_lightgbm(final_train, final_cat_cols)

# Bayesian Hyperparameter Optimisation (Optuna)

In [ ]:
def objective(trial):
    #define the search space dynamically
    params = {
        'objective': 'binary',
        'metric': 'average_precision',
        'scale_pos_weight': pos_weight, #from before
        'verbose': -1,
        'device':'cuda',

        #hyperparameters to tune
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 100, 1000),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 0.9),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 0.9),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True)}

    pruning_callback = LightGBMPruningCallback(trial, 'average_precision', valid_name='valid')

    #train the model using the trial's parameters
    model = lgb.train(
        params,
        dtrain,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'valid'],
        num_boost_round=1500,
        callbacks=[
            lgb.early_stopping(stopping_rounds=15, verbose=False),
            pruning_callback])

    #return the metric you want to optimize
    return model.best_score['valid']['average_precision']

#create the study and run 30 trials
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner(n_warmup_steps=20))
study.optimize(objective, n_trials=30)

print(f"Best Validation Average Precision: {study.best_value}")
print("Optimal Parameters:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value},")

In [ ]:
#extract winning hyperparameters from optuna
final_params = study.best_params

# Add back untuned structural parameters
final_params['objective'] = 'binary'
final_params['metric'] = 'average_precision'
final_params['scale_pos_weight'] = pos_weight
final_params['verbose'] = -1
final_params['device'] = 'cuda'

final_model, final_shap_importance, _ = run_lightgbm(final_train, final_cat_cols, final_params)